# 👁️ Convolutional Neural Network (CNN)
**Image Classification with PyTorch**
---

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print(f'PyTorch version : {torch.__version__}')
print('Libraries loaded ✅')

## 2. Load & Explore Dataset
> We use a **Synthetic 28x28 Grayscale Image** dataset containing three classes: Circles, Squares, and Triangles. This lightweight dataset perfectly demonstrates how CNNs learn spatial hierarchies without requiring massive downloads.

In [ ]:
df = pd.read_csv('../data/shapes_dataset.csv')
print(f'Shape   : {df.shape}')
print(f'Classes : {df["label"].value_counts().to_dict()}')

# Visualize a sample
pixel_cols = [c for c in df.columns if c.startswith('pixel_')]
sample_img = df.iloc[0][pixel_cols].values.reshape(28, 28)

plt.figure(figsize=(4, 4))
plt.imshow(sample_img, cmap='gray')
plt.title(f"Label: {df.iloc[0]['label']}", fontsize=14, fontweight='bold')
plt.axis('off')
plt.show()

## 3. Data Preprocessing
> CNNs in PyTorch expect input tensors in the shape `(batch_size, channels, height, width)`. For grayscale images, `channels = 1`. We also normalize pixel values to the `[0, 1]` range.

In [ ]:
X = df[pixel_cols].values.reshape(-1, 1, 28, 28).astype(np.float32) / 255.0
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
test_dataset = TensorDataset(torch.tensor(X_test), torch.tensor(y_test))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f'X_train shape: {X_train.shape}')  # (N, 1, 28, 28)
print(f'X_test shape : {X_test.shape}')

## 4. Build the CNN Model

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=3):
        super(SimpleCNN, self).__init__()
        # Block 1: Conv + ReLU + Pool
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1) # 28x28 -> 28x28
        self.pool1 = nn.MaxPool2d(2, 2)                         # 28x28 -> 14x14
        
        # Block 2: Conv + ReLU + Pool
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1) # 14x14 -> 14x14
        self.pool2 = nn.MaxPool2d(2, 2)                          # 14x14 -> 7x7
        
        # Classifier Head
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32 * 7 * 7, 64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.pool1(torch.relu(self.conv1(x)))
        x = self.pool2(torch.relu(self.conv2(x)))
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

model = SimpleCNN()
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f'\\nTotal Parameters: {total_params:,}')

## 5. Train the Model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 30
train_losses, val_accs = [], []

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        outputs = model(bx)
        loss = criterion(outputs, by)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * bx.size(0)
    
    train_losses.append(epoch_loss / len(train_dataset))

    # Validation
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for bx, by in test_loader:
            bx, by = bx.to(device), by.to(device)
            outputs = model(bx)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == by).sum().item()
            val_total += bx.size(0)
    
    val_accs.append(val_correct / val_total)
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{epochs} | Loss: {train_losses[-1]:.4f} | Val Acc: {val_accs[-1]:.4f}')

print('Training complete!')

## 6. Evaluate on Test Set

In [ ]:
model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for bx, by in test_loader:
        bx = bx.to(device)
        outputs = model(bx)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(by.cpu().numpy())

classes = ['Circle', 'Square', 'Triangle']
acc = accuracy_score(all_targets, all_preds)

print('='*50)
print('          CNN Test Set Results')
print('='*50)
print(f'  Accuracy: {acc:.4f}')
print('='*50)
print('\\nClassification Report:')
print(classification_report(all_targets, all_preds, target_names=classes))

## 7. Confusion Matrix

In [ ]:
cm = confusion_matrix(all_targets, all_preds)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=classes, yticklabels=classes,
            linewidths=1, linecolor='white')
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 8. Visualize Predictions

In [ ]:
# Get a batch of test data
dataiter = iter(test_loader)
images, labels = next(dataiter)
images, labels = images.to(device), labels.to(device)

model.eval()
with torch.no_grad():
    outputs = model(images)
    probs = torch.softmax(outputs, dim=1)
    _, preds = torch.max(outputs, 1)

# Plot first 6 images
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()

for i in range(6):
    img = images[i].cpu().squeeze().numpy()
    true_label = classes[labels[i].item()]
    pred_label = classes[preds[i].item()]
    conf = probs[i][preds[i]].item()
    
    axes[i].imshow(img, cmap='gray')
    color = '#10b981' if true_label == pred_label else '#f87171'
    axes[i].set_title(f"True: {true_label}\\nPred: {pred_label} ({conf:.1%})", color=color, fontweight='bold')
    axes[i].axis('off')

plt.tight_layout(); plt.show()

## 9. Save Model

In [ ]:
import os
os.makedirs('../models', exist_ok=True)
torch.save(model.state_dict(), '../models/cnn_model.pth')
print('Model saved → models/cnn_model.pth')

## 10. Key Takeaways
> - **Spatial Hierarchy**: CNNs learn simple features (edges) in early layers and complex features (shapes) in deeper layers.
> - **Parameter Efficiency**: Weight sharing in convolutional filters drastically reduces the number of parameters compared to dense networks.
> - **Translation Invariance**: Max pooling allows the network to recognize features regardless of their exact position in the image.
> - **Data Format**: Always remember to reshape image data to `(Batch, Channels, Height, Width)` for PyTorch Conv2d layers.